# TD 1 (suite) : Transformer le texte en nombres : Bag of Words et TF-IDF

> Ce notebook fait suite à la **section 6** du premier notebook (« Décrire une variable textuelle »).

On a vu comment décrire un texte avec des nombres simples : sa longueur, son nombre de mots, son vocabulaire.

Maintenant on veut aller plus loin : transformer **chaque avis entier** en une liste de nombres, pour qu'un modèle puisse s'en servir.

Deux méthodes très utilisées :

* **Bag of Words** : on compte simplement combien de fois chaque mot apparaît.
* **TF-IDF** : on fait la même chose, mais on donne moins de poids aux mots qu'on retrouve partout.

> **texte brut → nettoyage → nombres → analyse / modèle**

In [ ]:
import pandas as pd

DATA = "../data"

df = pd.read_csv(f"{DATA}/amazon_reviews_sample.csv")
avis = df["Text"].head(2000)

print("Nombre d'avis :", len(avis))
avis.head(3)

### Nettoyer le texte avant de compter les mots

Les avis Amazon contiennent parfois des balises HTML brutes, comme `<br />` pour les retours à la ligne. Si on ne les enlève pas, `CountVectorizer` les compte comme un mot à part entière. C'est pour ça que `br` se retrouvera en tête des mots les plus fréquents un peu plus loin, alors que ce n'est pas un vrai mot.

On les supprime avec une expression régulière, avant toute vectorisation.

In [ ]:
avis = avis.str.replace(r"<[^>]+>", " ", regex=True)

avis.head(3)

## 6.1 Bag of Words

### L'idée

Le **Bag of Words** (« sac de mots ») compte, pour chaque texte, combien de fois apparaît chaque mot du vocabulaire. On perd l'ordre des mots : seul le comptage compte.

Prenons un exemple tout simple, avec 3 petites phrases :

```text
"le chat est noir"
"le chien est noir"
"chat noir"
```

Le vocabulaire (tous les mots différents) est : `le, chat, est, noir, chien`.

Pour chaque phrase, on compte combien de fois apparaît chaque mot :

| | le | chat | est | noir | chien |
|---|---:|---:|---:|---:|---:|
| phrase 1 | 1 | 1 | 1 | 1 | 0 |
| phrase 2 | 1 | 0 | 1 | 1 | 1 |
| phrase 3 | 0 | 1 | 0 | 1 | 0 |

C'est exactement ce que fait `CountVectorizer` de scikit-learn. Exécutez la cellule et retrouvez ce tableau.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

phrases = [
    "le chat est noir",
    "le chien est noir",
    "chat noir",
]

vectorizer_demo = CountVectorizer()
X_demo = vectorizer_demo.fit_transform(phrases)

pd.DataFrame(
    X_demo.toarray(),
    columns=vectorizer_demo.get_feature_names_out()
)

### Le même principe sur les avis

On applique maintenant exactement la même idée aux **2 000 premiers avis**. Cette fois on ajoute `stop_words="english"`, pour ne pas compter les mots anglais qui n'apportent aucune information (`the`, `is`, `a`...).

`X_bow` contient une ligne par avis et une colonne par mot du vocabulaire.

In [ ]:
vectorizer = CountVectorizer(stop_words="english")
X_bow = vectorizer.fit_transform(avis)

print("Dimensions :", X_bow.shape)

Regardons maintenant quels mots reviennent le plus souvent, sur l'ensemble des 2 000 avis, en additionnant les colonnes de `X_bow`.

In [ ]:
frequences = pd.Series(
    X_bow.sum(axis=0).A1,
    index=vectorizer.get_feature_names_out()
)

frequences.sort_values(ascending=False).head(15)

Sans surprise, on retrouve des mots comme `product`, `food`, `taste`... Ils sont fréquents dans presque tous les avis, bons comme mauvais, donc ils ne nous disent pas grand-chose sur un avis en particulier. C'est exactement ce problème que le TF-IDF va corriger.

## 6.2 TF-IDF

### L'idée

Le TF-IDF reprend le même comptage, mais donne **moins de poids** aux mots qui apparaissent dans beaucoup d'avis, et **plus de poids** aux mots qui apparaissent surtout dans un avis en particulier.

```text
poids(mot, avis) = fréquence du mot dans cet avis  ×  rareté du mot dans l'ensemble des avis
```

* Un mot présent dans presque tous les avis (`food`, `product`) → poids faible.
* Un mot rare dans l'ensemble, mais répété dans un avis → poids fort.

Reprenons notre exemple des chats et chiens :

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_demo = TfidfVectorizer()
X_tfidf_demo = tfidf_demo.fit_transform(phrases)

pd.DataFrame(
    X_tfidf_demo.toarray(),
    columns=tfidf_demo.get_feature_names_out()
).round(2)

On remarque que `noir`, présent dans les **3** phrases, a un poids plus faible que `chat` ou `chien`, qui n'apparaissent que dans 2 phrases sur 3 (voire une seule pour `chien`). Le mot `noir` est partout, donc il compte moins : il n'aide pas à distinguer les phrases entre elles.

### Le même principe sur les avis

On applique `TfidfVectorizer` (toujours avec `stop_words="english"`) sur les mêmes 2 000 avis, puis on regarde le premier avis en détail : quels sont ses mots avec le plus haut score TF-IDF ?

In [ ]:
tfidf_vectorizer = TfidfVectorizer(stop_words="english")
X_tfidf = tfidf_vectorizer.fit_transform(avis)

print("Dimensions :", X_tfidf.shape)
print("\nAvis n°0 :\n", avis.iloc[0])

In [ ]:
# Mots avec le plus haut score TF-IDF pour le premier avis
avis_0_tfidf = pd.Series(
    X_tfidf[0].toarray().flatten(),
    index=tfidf_vectorizer.get_feature_names_out()
)

avis_0_tfidf.sort_values(ascending=False).head(10)

In [ ]:
# Mots les plus comptés (Bag of Words) pour le même avis, à comparer
avis_0_bow = pd.Series(
    X_bow[0].toarray().flatten(),
    index=vectorizer.get_feature_names_out()
)

avis_0_bow.sort_values(ascending=False).head(10)

En Bag of Words, ce sont souvent des mots génériques (fréquents dans tout le corpus) qui arrivent en tête. En TF-IDF, ce sont plutôt des mots **propres à cet avis** qui ressortent. C'est là toute la différence entre les deux représentations.

### À retenir

* **Bag of Words** : on compte les mots, tout simplement. Simple, mais les mots fréquents partout prennent trop de place.
* **TF-IDF** : on compte aussi les mots, mais on réduit le poids de ceux qu'on retrouve dans presque tous les avis. Les mots vraiment caractéristiques d'un avis ressortent mieux.

> **Bag of Words = comptage. TF-IDF = comptage pondéré par la rareté.**

Dans les deux cas, chaque avis devient un **vecteur de nombres**, qu'un modèle de machine learning peut ensuite utiliser (classification, clustering, etc.).

### À vous : quelle représentation choisir ?

Pour chacun des deux cas ci-dessous, choisissez la représentation la plus adaptée : **Bag of Words** ou **TF-IDF**.

**Cas 1 : mesurer l'enthousiasme des clients**

Vous voulez compter, pour chaque avis, combien de fois apparaissent des mots comme `love`, `great`, `amazing`. Plus un avis répète ces mots, plus il est considéré comme enthousiaste. Peu importe que ces mots soient aussi très fréquents dans les autres avis.

**Cas 2 : moteur de recherche**

Un employé tape une requête courte, par exemple `"gluten free cookies"`, et l'outil doit retrouver, parmi des milliers d'avis, ceux qui parlent vraiment de ce sujet précis, pas seulement ceux qui contiennent des mots génériques très fréquents.

**Question.** Pour chaque cas, quelle méthode choisiriez-vous, et pourquoi ce n'est pas la même dans les deux cas ?

# Fin